In [23]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ['GOOGLE_API_KEY'] = os.getenv("GOOGLE_API_KEY")
os.environ["LANGSMITH_TRACING"] = 'true'
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

In [24]:
from langchain.chat_models import init_chat_model

groq_llm = init_chat_model("groq:openai/gpt-oss-120b")
groq_llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F1DD222E90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F1DD223890>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [25]:
response = groq_llm.invoke("How to learn fastapi")
response

AIMessage(content='## Quick‑Start Road‑Map to Master FastAPI 🚀\n\nBelow is a **step‑by‑step learning plan** you can follow in a few weeks, plus a curated list of **free & paid resources**, **hands‑on projects**, and **best‑practice tips**. Feel free to cherry‑pick the parts that fit your schedule and background.\n\n---\n\n### 1️⃣ Get the Basics Right\n\n| Goal | What to Do | Resources |\n|------|------------|-----------|\n| **Python foundation** | Be comfortable with Python 3.8+ (type hints, `async`/`await`). | •\u202fReal‑Python “Python Type Hints” <br>•\u202fCorey Schafer’s async tutorials (YouTube) |\n| **HTTP & REST** | Understand verbs, status codes, headers, query‑/path‑params, JSON payloads. | •\u202fMDN “HTTP Overview” <br>•\u202fRESTful API design guide (freecodecamp) |\n| **ASGI vs WSGI** | Know why FastAPI runs on ASGI (async server gateway interface). | •\u202fFastAPI docs “What is ASGI?” <br>•\u202fStarlette intro (FastAPI’s underlying toolkit) |\n\n---\n\n### 2️⃣ Install 

In [32]:
from langchain.tools import  tool
import pytz
from datetime import datetime

@tool
def get_datetime(city: str) -> str:
    """Get the current date and time for a city."""
    
    city_timezones = {
        "india": "Asia/Kolkata",
        "chennai": "Asia/Kolkata",
        "delhi": "Asia/Kolkata",
        "new york": "America/New_York",
        "london": "Europe/London"
    }

    tz = city_timezones.get(city.lower())

    if not tz:
        return f"Sorry, I don't know the timezone for {city}."

    timezone = pytz.timezone(tz)
    current_time = datetime.now(timezone)

    return f"The current date and time in {city} is {current_time.strftime('%Y-%m-%d %H:%M:%S')}."

@tool
def get_weather(city:str)-> str:
    """Get the weather for a city."""
    return f"The weather in {city} is sunny."

In [27]:
model_with_tools = groq_llm.bind_tools([get_datetime,get_weather])
model_with_tools

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F1DD222E90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F1DD223890>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_datetime', 'description': 'Get the current date and time for a city.', 'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather for a city.', 'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}}}]}, co

In [28]:
response_weather = model_with_tools.invoke("What is the weather like in pondicherry?")
response_weather
for tool_calls in response_weather.tool_calls:
    print(f"Tools: {tool_calls['name']}")
    print(f"Args: {tool_calls['args']}")

Tools: get_weather
Args: {'city': 'Pondicherry'}


In [29]:
response_datetime = model_with_tools.invoke("What is date and time in chennai?")
response_datetime
for tool_calls in response_datetime.tool_calls:
    print(f"Tools: {tool_calls['name']}")
    print(f"Args: {tool_calls['args']}")

Tools: get_datetime
Args: {'city': 'Chennai'}


In [30]:
### Tools execution loops


messages = [{"role": "user", "content": "What is the weather like in pondicherry?"}]

ai_msg = model_with_tools.invoke(messages)

messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)

print(final_response.text)
print("----------------------------")
print(messages)

The current weather in Pondicherry is sunny. Let me know if you’d like more details such as temperature, humidity, or a forecast for the next few days!
----------------------------
[{'role': 'user', 'content': 'What is the weather like in pondicherry?'}, AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to get weather for Pondicherry. Use function get_weather.', 'tool_calls': [{'id': 'fc_9ead4d0e-5edd-473a-b5fb-47a72e23f284', 'function': {'arguments': '{"city":"Pondicherry"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 155, 'total_tokens': 201, 'completion_time': 0.11143639, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.005976437, 'prompt_tokens_details': None, 'queue_time': 0.045289833, 'total_time': 0.117412827}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_d29d1d1418', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs':

In [33]:
messages = [{"role": "user", "content": "What is date and time in chennai?"}]

ai_msg = model_with_tools.invoke(messages)

messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_datetime.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)

print(final_response.text)
print("----------------------------")
print(messages)

The current date and time in Chennai is **2026‑03‑17 23:03 (UTC+5:30)**.
----------------------------
[{'role': 'user', 'content': 'What is date and time in chennai?'}, AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks for date and time in Chennai. Use function get_datetime.', 'tool_calls': [{'id': 'fc_418b292d-964f-4633-b649-fa33266b74a6', 'function': {'arguments': '{"city":"Chennai"}', 'name': 'get_datetime'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 154, 'total_tokens': 197, 'completion_time': 0.11080442, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.007984248, 'prompt_tokens_details': None, 'queue_time': 0.047912882, 'total_time': 0.118788668}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a09bde29de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cfcdc-7cc8-7143-aba1-c549eabfa965-0', t